In [ ]:
import os

import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray as rioxr

from sklearn.metrics import confusion_matrix

# Class labels used in the model raster and the validation points
CLASSES = {1: 'iceplant', 2: 'other', 3: 'other vegetation'}

In [2]:
# Validation points
pts = gpd.read_file(os.path.join('data','2025-sample-points','validation_sample_points_2025.shp'))
pts

,class,raster,LON,LAT,eva,geometry
0,3,2025_rawmodel.tif,-120.457684,34.449840,3,POINT (-120.45768 34.44984)
1,1,2025_rawmodel.tif,-120.472756,34.472097,1,POINT (-120.47276 34.47210)
2,1,2025_rawmodel.tif,-120.464888,34.465340,1,POINT (-120.46489 34.46534)
3,3,2025_rawmodel.tif,-120.454276,34.460803,3,POINT (-120.45428 34.46080)
4,1,2025_rawmodel.tif,-120.466544,34.459381,1,POINT (-120.46654 34.45938)
...,...,...,...,...,...,...
707,1,2025_rawmodel.tif,-120.468722,34.464030,1,POINT (-120.46872 34.46403)
708,3,2025_rawmodel.tif,-120.452600,34.459870,3,POINT (-120.45260 34.45987)
709,1,2025_rawmodel.tif,-120.467996,34.462554,1,POINT (-120.46800 34.46255)
710,1,2025_rawmodel.tif,-120.464310,34.461564,1,POINT (-120.46431 34.46156)


In [ ]:
# Number of points per class and map/reviewer classification
pd.DataFrame({'map': pts['class'].value_counts(),
              'reviewed': pts['eva'].value_counts()}).rename(index=CLASSES)

In [ ]:
# Classification results
# 0 is the raster's nodata value (_FillValue), so mask it
raster = rioxr.open_rasterio(os.path.join('data','2025_rawmodel_matched.tif'), masked=True).squeeze(drop=True)
raster

In [ ]:
# Labels extracted from raster via QGIS
# 0 = nodata (masked)
# 1 = iceplant
# 2 = other
# 3 = other vegetation

# Number of pixels per class, excluding nodata
values = raster.values[~np.isnan(raster.values)].astype(int)
classes, counts = np.unique(values, return_counts=True)
stats = pd.DataFrame({'name': [CLASSES[c] for c in classes], 'n': counts},
                     index=pd.Index(classes, name='class'))
stats

In [ ]:
# Calculate area coverage per class: fraction, percentage and hectares

x_res, y_res = raster.rio.resolution()
pixel_area = abs(x_res * y_res) * 0.0001 # Convert from meters squared to hectares

total_pixels = stats['n'].sum()
total_area = total_pixels * pixel_area

stats['fraction'] = stats['n']/total_pixels
stats['perc coverage'] = stats['fraction']*100
stats['hectares'] = total_area * stats['fraction']
stats

In [ ]:
# Calculate confusion matrix

# n_ij in this matrix = classified as i on the map, real class is j (Olofsson convention)
n = confusion_matrix(pts['class'], pts['eva'], labels=list(CLASSES))
pd.DataFrame(n,
             index=pd.Index(list(CLASSES.values()), name='map'),
             columns=pd.Index(list(CLASSES.values()), name='reviewed'))

In [ ]:
# Fraction of area estimation
n_idot = [n[i].sum() for i in range(3)]

def calculate_p_dotj(j, n):
    # j = the class 1=iceplant, 2=other, 3=other vegetation
    return stats.at[1, 'fraction']*n[0,j-1]/n_idot[0] + stats.at[2, 'fraction']*n[1,j-1]/n_idot[1] + stats.at[3, 'fraction']*n[2,j-1]/n_idot[2]

stats['adj fraction'] = [calculate_p_dotj(j,n) for j in [1,2,3]]

stats['adj perc'] = stats['adj fraction']*100
stats['adj hectares'] = total_area * stats['adj fraction']
stats

In [ ]:
# Double check these add up to 1
stats['adj fraction'].sum()

In [ ]:
diff = stats.at[1,'adj hectares'] - stats.at[1,'hectares']

print(f'Points mapped as OVG but reviewed as iceplant: {n[2,0]}')
print(f'Points mapped as other but reviewed as iceplant: {n[1,0]}')
print(f'The diff between the estimated iceplant area and the iceplant map area is {diff:.2f} ha')
print(f'So each point amounts for {diff/(n[2,0]+n[1,0]):.2f} ha')

What next?

Thoughts:
- 3 points is roughly the size of the false poistive patch (6 ha). 
- From a general view at the 2025 model results, my sense is this is overestimating. I disagree with how some of the review points were classified. We probably need to redo this accuracy assessment. 

Next steps:
- Investigate why the 2025 model results look so different from previous years
- Check Zhe's code (though the numerical results are pretty similar)
- Read manuscript and see what is the issue here: is it the framing? Probably we cannot attribute these results to iceplant dynamics, and a lot of these changes have to do with sensros, although we can track specific locations that underwent restoration. 